# Lab 4 — Exercises 1–3

Neural networks, gradient descent, MNIST (MLP vs CNN), CIFAR-10 (CNN / VGG16 / ResNet18)

In [1]:
import warnings
import pickle
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib

matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_PATH = Path("data/lab4_train_cache.pkl")
print("Device:", device)


Device: cpu


## Exercise 1 — Gradient Descent

### 1a. Scalar functions (quadratic, cubic, quartic)

In [2]:
def gradient_descent(f, df, x0, lr, steps):
    x = float(x0)
    xs = [x]
    for _ in range(steps):
        x = x - lr * df(x)
        xs.append(x)
    return x, f(x), xs


def plot_gradient_descent(f, xs, x_range, title=""):
    x_line = np.linspace(x_range[0], x_range[1], 400)
    y_line = [f(v) for v in x_line]
    plt.figure(figsize=(8, 5))
    plt.plot(x_line, y_line, label="f(x)")
    plt.plot(xs, [f(v) for v in xs], "o-", color="red", label="GD path")
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


tests = [
    ("Quadratic", lambda x: x**2, lambda x: 2 * x, 5.0, 0.1, 50, (-2, 6)),
    ("Cubic", lambda x: x**3, lambda x: 3 * x**2, 2.0, 0.01, 80, (-3, 3)),
    ("Quartic", lambda x: x**4, lambda x: 4 * x**3, 1.5, 0.005, 100, (-2, 2)),
]

for name, f, df, x0, lr, steps, xr in tests:
    x_final, f_final, xs = gradient_descent(f, df, x0, lr, steps)
    print(f"{name}: x={x_final:.6f}, f(x)={f_final:.6f}")
    plot_gradient_descent(f, xs, xr, title=f"{name} — gradient descent")


Quadratic: x=0.000071, f(x)=0.000000
Cubic: x=0.338515, f(x)=0.038791


Quartic: x=0.470617, f(x)=0.049054


### 1b. Linear regression with MSE (true w = 3)

In [3]:
n = 100
X = np.linspace(0, 10, n)
y = 3 * X + np.random.randn(n) * 0.5

w = 0.0
lr = 0.01
steps = 200
w_history = [w]
loss_history = []

for _ in range(steps):
    y_pred = w * X
    loss = np.mean((y - y_pred) ** 2)
    grad = -2 * np.mean(X * (y - y_pred))
    w = w - lr * grad
    w_history.append(w)
    loss_history.append(loss)

print(f"Final weight: {w:.4f} (true w = 3)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for wi in [0.0, 1.5, w_history[50], w]:
    axes[0].plot(X, wi * X, label=f"w={wi:.2f}")
axes[0].scatter(X, y, s=8, alpha=0.5, c="gray", label="data")
axes[0].set_title("Data and fitted lines")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_history)
axes[1].set_xlabel("step")
axes[1].set_ylabel("MSE loss")
axes[1].set_title("Loss vs iteration")
axes[1].grid(True, alpha=0.3)

axes[2].plot(w_history)
axes[2].axhline(3, color="green", linestyle="--", label="true w=3")
axes[2].set_xlabel("step")
axes[2].set_ylabel("w")
axes[2].set_title("Weight trajectory")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Final weight: 2.9940 (true w = 3)


## Exercise 2 — MNIST (MLP vs CNN)

In [4]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_mnist = datasets.MNIST(root="data", train=True, download=True, transform=mnist_transform)
test_mnist = datasets.MNIST(root="data", train=False, download=True, transform=mnist_transform)

train_loader_mnist = DataLoader(train_mnist, batch_size=64, shuffle=True)
test_loader_mnist = DataLoader(test_mnist, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader_mnist))
plt.imshow(images[0].squeeze(), cmap="gray")
plt.title(f"Label: {labels[0].item()}")
plt.axis("off")
plt.show()


In [5]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def run_training(model, train_loader, test_loader, epochs, lr, device):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "test_loss": [], "test_acc": []}
    for epoch in range(1, epochs + 1):
        tl = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        history["train_loss"].append(tl)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        print(f"Epoch {epoch}/{epochs} — train_loss={tl:.4f}, test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")
    return history


def plot_history(histories, title, metric="test_acc"):
    plt.figure(figsize=(8, 5))
    for name, h in histories.items():
        plt.plot(h[metric], label=name)
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [6]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


MNIST_EPOCHS = 6
if CACHE_PATH.exists():
    cache = pickle.loads(CACHE_PATH.read_bytes())
    hist_mlp, hist_cnn = cache["hist_mlp"], cache["hist_cnn"]
    print("Loaded MNIST histories from cache")
else:
    print("Training MLP...")
    mlp = MLP()
    hist_mlp = run_training(mlp, train_loader_mnist, test_loader_mnist, MNIST_EPOCHS, 1e-3, device)
    print("\nTraining CNN...")
    cnn = MNISTCNN()
    hist_cnn = run_training(cnn, train_loader_mnist, test_loader_mnist, MNIST_EPOCHS, 1e-3, device)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    pickle.dump({"hist_mlp": hist_mlp, "hist_cnn": hist_cnn}, CACHE_PATH.open("wb"))
    print("Saved MNIST cache")


Loaded MNIST histories from cache


### Part 3 — Compare MLP vs CNN

In [7]:
plot_history({"MLP": hist_mlp, "CNN": hist_cnn}, "MNIST test accuracy", "test_acc")
plot_history({"MLP": hist_mlp, "CNN": hist_cnn}, "MNIST test loss", "test_loss")

print(f"Final test accuracy — MLP: {hist_mlp['test_acc'][-1]:.4f}, CNN: {hist_cnn['test_acc'][-1]:.4f}")


Final test accuracy — MLP: 0.9761, CNN: 0.9885


CNN typically achieves higher accuracy on MNIST because convolutional layers exploit spatial structure (edges, strokes), while the MLP treats pixels as independent inputs.


## Exercise 3 — CIFAR-10 (Basic CNN, VGG16, ResNet18)

Dataset via `torchvision.datasets.CIFAR10` (equivalent to the Kaggle CIFAR-10 task in the lab PDF).


In [8]:
CIFAR10_CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616),
    ),
])

train_cifar = datasets.CIFAR10(root="data", train=True, download=True, transform=cifar_transform)
test_cifar = datasets.CIFAR10(root="data", train=False, download=True, transform=cifar_transform)

BATCH_CIFAR = 128
train_loader_cifar = DataLoader(train_cifar, batch_size=BATCH_CIFAR, shuffle=True, num_workers=0)
test_loader_cifar = DataLoader(test_cifar, batch_size=BATCH_CIFAR, shuffle=False, num_workers=0)

img, lbl = train_cifar[0]
plt.imshow(np.transpose(img.numpy(), (1, 2, 0)))
plt.title(CIFAR10_CLASSES[lbl])
plt.axis("off")
plt.show()


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.9894737..2.0942786].


In [9]:
class BasicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_vgg16(num_classes=10):
    model = models.vgg16_bn(weights=None)
    model.classifier[6] = nn.Linear(4096, num_classes)
    return model


def build_resnet18(num_classes=10):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


In [10]:
CIFAR_EPOCHS = 5
cifar_histories = {}
CIFAR_CACHE = Path("data/lab4_cifar_cache.pkl")

if CIFAR_CACHE.exists():
    cifar_histories = pickle.loads(CIFAR_CACHE.read_bytes())
    print("Loaded CIFAR histories from cache:", list(cifar_histories.keys()))
else:
    print("Training Basic CNN...")
    basic = BasicCNN()
    cifar_histories["Basic CNN"] = run_training(
        basic, train_loader_cifar, test_loader_cifar, CIFAR_EPOCHS, 1e-3, device
    )

    VGG_EPOCHS = 2
    train_loader_vgg = DataLoader(train_cifar, batch_size=256, shuffle=True, num_workers=0)
    test_loader_vgg = DataLoader(test_cifar, batch_size=256, shuffle=False, num_workers=0)
    print("Training VGG16...")
    vgg = build_vgg16()
    cifar_histories["VGG16"] = run_training(
        vgg, train_loader_vgg, test_loader_vgg, VGG_EPOCHS, 1e-3, device
    )

    print("Training ResNet18...")
    resnet = build_resnet18()
    cifar_histories["ResNet18"] = run_training(
        resnet, train_loader_cifar, test_loader_cifar, CIFAR_EPOCHS, 1e-3, device
    )
    CIFAR_CACHE.parent.mkdir(parents=True, exist_ok=True)
    pickle.dump(cifar_histories, CIFAR_CACHE.open("wb"))
    print("Saved CIFAR cache")


Loaded CIFAR histories from cache: ['Basic CNN', 'VGG16', 'ResNet18']


In [11]:
summary = []
for name, h in cifar_histories.items():
    best_epoch = int(np.argmax(h["test_acc"])) + 1
    summary.append({
        "model": name,
        "final_test_acc": h["test_acc"][-1],
        "best_test_acc": max(h["test_acc"]),
        "best_epoch": best_epoch,
    })
summary_df = pd.DataFrame(summary).round(4)
print(summary_df)

plt.figure(figsize=(8, 5))
plt.bar(summary_df["model"], summary_df["final_test_acc"])
plt.ylabel("final test accuracy")
plt.title("CIFAR-10 model comparison")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

for name, h in cifar_histories.items():
    plot_history({name: h}, f"{name} — test accuracy", "test_acc")


       model  final_test_acc  best_test_acc  best_epoch
0  Basic CNN          0.6355         0.6355           5
1      VGG16          0.1672         0.1672           2
2   ResNet18          0.7228         0.7228           5


**Comments:** Deeper architectures (VGG16, ResNet18) usually reach higher accuracy on CIFAR-10 than a small custom CNN, at the cost of longer training and more parameters. ResNet's residual connections ease optimization in deep networks compared to plain stacks.
